## Prompt Chaining

---

**Description:**  
Prompt chaining is the process of connecting multiple prompts together so that the output of one step becomes the input to the next. Instead of relying on a single, complex prompt, tasks are decomposed into smaller steps, making them easier to manage, debug, and optimize. This approach allows large language models (LLMs) to build context progressively and produce higher-quality results.

---

**When to Use:**  
- When a task is **too complex for a single prompt** (e.g., generating a long script, then summarizing it).  
- When you need to **add context step by step**, such as turning a topic into a title, then a description, then a post.  
- When different stages of a workflow need **different prompt styles** (creative, factual, summarizing).  
- When you want **more control and interpretability** over each stage of generation.  
- When results from one step should be **repurposed downstream**, like using a research summary to generate exam questions.  


In [2]:
from typing import TypedDict, Dict
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from utils import get_groq_llm

In [3]:
llm = get_groq_llm()

In [5]:
class YouTubeWorkflowState(TypedDict):
    topic: str
    title: str | None
    description: str | None
    script: str | None
    linkedin_post: str | None
    hashtags: str | None
    final_output: Dict | None

In [6]:
title_prompt = ChatPromptTemplate.from_template(
"""
Write a short, catchy YouTube video title for the topic: "{topic}".
Keep it between 50 and 70 characters, include a clear value proposition or hook.
Respond with only the title (no extra text).
"""
)

description_prompt = ChatPromptTemplate.from_template(
"""
Write a YouTube video description for the video titled: "{title}" about "{topic}".
Include a 2-3 sentence summary, 3 key takeaways, and a call-to-action to subscribe and watch.
Keep it under 300 words and respond only with the description text.
"""
)

script_prompt = ChatPromptTemplate.from_template(
"""
Write a detailed spoken script for a ~20-minute YouTube video using the following inputs:
- Title: {title}
- Topic: {topic}
- Description: {description}


Structure the script with timestamps (approximate minutes), section headings, and speaker text.
Aim for natural spoken language, include examples and transitions. The final output should be a single script that
can be read to fill ~20 minutes of spoken content. Do NOT include stage directions or editor notes.
Respond with the full script only.
"""
)

linkedin_prompt = ChatPromptTemplate.from_template(
"""
I published a YouTube video titled "{title}" about "{topic}".
Write a LinkedIn post that:
- Announces the video (1-2 sentences)
- Shares 3 insightful takeaways or facts from the video
- Encourages professionals to watch and discuss
Keep the post professional and around 120-180 words. Respond with the post only.
"""
)

hashtag_prompt = ChatPromptTemplate.from_template(
"""
Generate 30 relevant hashtags for a YouTube video about "{topic}" with title "{title}".
Return them as a comma-separated list (no extra commentary). Keep hashtags short and relevant.
"""
)

In [9]:
def gen_title(state: YouTubeWorkflowState):
    prompt_input = {"topic": state["topic"]}
    response = (title_prompt | llm).invoke(prompt_input)
    title_text = response.content.strip()
    return {"title": title_text}

def gen_description(state: YouTubeWorkflowState):
    prompt_input = {"title": state.get("title", ""), "topic": state["topic"]}
    response = (description_prompt | llm).invoke(prompt_input)
    return {"description": response.content.strip()}

def gen_script(state: YouTubeWorkflowState):
    prompt_input = {
    "title": state.get("title", ""),
    "topic": state["topic"],
    "description": state.get("description", ""),
    }
    response = (script_prompt | llm).invoke(prompt_input)
    return {"script": response.content.strip()}

def gen_linkedin(state: YouTubeWorkflowState):
    prompt_input = {
    "title": state.get("title", ""),
    "topic": state["topic"],
    "description": state.get("description", ""),
    }
    response = (linkedin_prompt | llm).invoke(prompt_input)
    return {"linkedin_post": response.content.strip()}

def gen_hashtags(state: YouTubeWorkflowState):
    prompt_input = {"title": state.get("title", ""), "topic": state["topic"]}
    response = (hashtag_prompt | llm).invoke(prompt_input)
    hashtags = response.content.strip().replace("\n", " ")
    return {"hashtags": hashtags}

def merge(state: YouTubeWorkflowState):
    return {
        "final_output": {
        "title": state.get("title"),
        "description": state.get("description"),
        "script": state.get("script"),
        "linkedin_post": state.get("linkedin_post"),
        "hashtags": state.get("hashtags"),
        }
    }

In [12]:
builder = StateGraph(YouTubeWorkflowState)

builder.add_node("title", gen_title)
builder.add_node("description", gen_description)
builder.add_node("script", gen_script)
builder.add_node("linkedin", gen_linkedin)
builder.add_node("hashtags", gen_hashtags)
builder.add_node("merge", merge)

builder.add_edge("__start__", "title")
builder.add_edge("title", "description")
builder.add_edge("description", "script")
builder.add_edge("script", "linkedin")
builder.add_edge("linkedin", "hashtags")
builder.add_edge("hashtags", "merge")
builder.add_edge("merge", END)

graph = builder.compile()

In [14]:
topic_input = "Technical Analysis"

result = graph.invoke({"topic": topic_input})

final = result.get("final_output")

print("\n--- TITLE ---\n")
print(final.get("title"), end="\n\n")

print("--- DESCRIPTION ---\n")
print(final.get("description"), end="\n\n")

print("--- SCRIPT (truncated preview - first 800 chars) ---\n")
print(final.get("script"), end="\n\n")

print("--- LINKEDIN POST ---\n")
print(final.get("linkedin_post"), end="\n\n")

print("--- HASHTAGS ---\n")
print(final.get("hashtags"))


--- TITLE ---

"Master Charts, Dominate Markets: Unlock Technical Analysis Secrets"

--- DESCRIPTION ---

**Master Charts, Dominate Markets: Unlock Technical Analysis Secrets**

In this comprehensive guide, we'll delve into the world of technical analysis and reveal the secrets to mastering charts and dominating markets. You'll learn how to identify trends, predict price movements, and make informed investment decisions with confidence. Whether you're a seasoned trader or just starting out, this video will equip you with the skills and knowledge you need to succeed in the fast-paced world of finance.

**Key Takeaways:**

 1. **Learn to read charts like a pro**: Discover how to interpret technical indicators, identify patterns, and make sense of market trends.
 2. **Understand market psychology**: Explore the mindset of traders and investors, and learn how to use this knowledge to your advantage.
 3. **Develop a winning trading strategy**: Create a personalized approach that suits your